# 4.6 Your turn: distributions in your own chat

[04.1](04.1-parametric-distributions.ipynb) through [04.5](04.5-fitting-the-residual.ipynb)
built the six-family toolkit — the registry, `DistributionFitter`, `fit_table`'s two
winners, `PlotFits`, `QQPlot`, `ECDFPlot` — on data chosen so a shape was guaranteed to be
there. This notebook makes the same move [03.3](../lesson3/roos_03.3-events-in-your-chat.ipynb)
and [02.5](../lesson2/roos_02.5-your_turn.ipynb) already made: your data, your question,
and nothing guaranteed.

**Notebook 4.6 is split in four:** [a: move and Benji](roos_04.6a-move-and-baby.ipynb) (§4.6.1–4.6.9) · [b: symptoms and topics](roos_04.6b-symptoms-and-topics.ipynb) (§4.6.10–4.6.11) · [c: graded visualisation 3](roos_04.6c-graded-vis3-positive-test.ipynb) (§4.6.12) · [d: the hard weeks](roos_04.6d-hard-weeks.ipynb) (§4.6.13). Loading, the four dates, `save` and the helpers §4.6.13 reuses live in [roos_046_shared.py](roos_046_shared.py).

**x-axis: the value. y-axis: how likely it is** — a density for something continuous
(seconds, characters), a probability mass for something discrete (a count). *Which*
variable, and *which* family, is what the rest of this notebook works out with you rather
than for you.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.analytics import DistributionFitter, FitResult, NullDistribution
from goad_toolkit.datatransforms import Pipeline, TimeFeatures, TransformBase
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import (
    DistPlot,
    ECDFPlot,
    HorizontalLine,
    NullPlot,
    PlotSettings,
)
from roos_046_shared import (
    BABY_DATE,
    MOVE_DATE,
    load_own,
)

from scripts.plots import BarPlot
from roos_pipeline.plots import DistributionDiffPlot


## 4.6.1 Loading your data

Same rule as [02.5](../lesson2/roos_02.5-your_turn.ipynb): `load_own_chat()` raises when
there is no export loaded, on purpose — everything below is about *your* messages, not a
showcase dataset with a shape already built in.

In [ ]:
own = load_own()
own.head()

## 4.6.2 What's already loaded

From `scripts/plots.py`: `BarPlot`, `BarPlotWithError` (02.2/02.4), `EventMarker` and
`RiseDeclinePlot` (03.3, the latter colouring a line by whether it's rising or falling).
None of them fit a distribution — that's `goad_toolkit`'s job, imported above:
`DistributionRegistry` + `DistributionFitter` do the fitting, `fit_table` ranks the
results, and `HistogramPlot` / `DistPlot` / `QQPlot` / `ECDFPlot` / `PlotFits` draw it.
All of them are `BasePlot`s too, so they compose with `plot_on` / `plot_on_axes` /
`create_figure(n_plots=...)` exactly like the classes in `scripts/plots.py` do — if a
chart this notebook needs doesn't exist in either place yet, a new `BasePlot` subclass in
`scripts/plots.py` is where it belongs, the same way `RiseDeclinePlot` got added there for
03.3.

## 4.6.3 Brainstorming with `goad`

Same tools as [02.5](../lesson2/roos_02.5-your_turn.ipynb#2.5.3), plus one built for
exactly this notebook:

- **`goad_distribution_method`** — the order of operations for fitting anything: count
  first, look before summarising, decide discrete-or-continuous, say what you expect
  *before* fitting, fit the whole registry rather than one family, and only then decide
  what counts as an outlier. Worth running before the first `DistributionFitter.fit()`
  call below, not after.
- **`goad_analysis_checklist`** — the six-stage interview (question, data, shape,
  encoding, critique, verification), one stage at a time, in your own words.
- **`goad_search`** / **`goad_list_concepts`** / **`goad_get_concept`** — look up a
  concept directly, e.g. `goad_get_concept('05')` for the full distributions chapter.
- **`goad_critique_visual`** — once there's a chart here to defend.

> **Your turn.**
>
> Before any `DistributionFitter.fit()` call — same order `goad_distribution_method`
> gives, steps 1-4 — answer these in your own words:
>
> 1. **Which variable, and why does it matter to you?** Not "what looks curious" but
>    what question about your own days or your own relationship it would actually answer.
> 2. **How many independent observations does it really rest on?** Messages are not
>    people — say both.
> 3. **Discrete or continuous?**
> 4. **What do you expect, before looking** — which family, and what mechanism would
>    produce it (a sum, a product, a steady rate, a rank)?
>
> **My answers:**
>
> 1. Response time — mine and Louis's, split before/after his move to NL (Nov 2024).
>    Before the move I might have had an overall lower response time and Louis a faster
>    one; after the move there might be a switch. Not sure if that's right — I'm used to
>    thinking about this in a time series, not a distribution.
> 2. Only the *close* response times should count — at night we sleep, and a reply after
>    hours of silence isn't really "a response time", it's "woke up". (Handled below by
>    excluding responses over a duration cutoff before doing anything else — see 4.6.4.)
> 3. Continuous (seconds).
> 4. Predicted the *outcome* (who's faster, before vs. after) above; haven't committed to
>    a specific family yet — that's still open.

## 4.6.4 Response time, before vs. after the move — first look, no fitting yet

Per §4.3 of the "five families" chapter: "comparing two groups by their histograms alone"
is one of distributions' classic ways of lying, when the groups differ in size — normalise,
or use ECDFs. This is that: four ECDFs (Roos/Louis × before/after) on one bin-free axis, no
family fitted yet — just looking at the shape, per `goad_distribution_method` step 2.

Two pieces reused rather than rebuilt:

- **`response_seconds`** — the same `ResponseTimeFeature` transform from `roos_01.4`:
  seconds to a message from the *other* author, `NaN` for same-author bursts (so the
  correlated-burst problem your own analysis-log already flagged for this variable stays
  visible, not fitted around).
- **`MOVE_DATE = "2024-11-01"`** — the same constant `roos_02.5` uses for Louis's move.

**One new choice, mine, not yours**: excluding responses over `MAX_RESPONSE_MINUTES`
(currently 4 hours) as "asleep, not a slow reply" — 04.4's own gap-fitting cell uses a
1-hour cutoff for *any* message gap ("a burst, not overnight"), but a response can take
longer than that and still be a real response, so 4 hours is more generous. Change the
constant and rerun if that doesn't match what you'd call "still a response".

In [ ]:
class ResponseTimeFeature(TransformBase):
    """Seconds to this message from the previous one, only when the author changed --
    i.e. how long the responder took to reply. Same-author bursts get NaN.
    (Unchanged from roos_01.4.)"""

    def transform(
        self,
        data: pd.DataFrame,
        timestamp_col: str = "timestamp",
        author_col: str = "author",
        feature: str = "response_seconds",
    ) -> pd.DataFrame:
        ts = pd.to_datetime(data[timestamp_col], utc=True).dt.tz_convert(None)
        gap = ts.diff().dt.total_seconds()
        same_author = data[author_col] == data[author_col].shift(1)
        data[feature] = gap.where(~same_author)
        return data


MAX_RESPONSE_MINUTES = 240  # see the note above -- adjust and rerun to taste

own_response = Pipeline().add(ResponseTimeFeature).apply(own)
own_response["timestamp"] = own_response["timestamp"].dt.tz_convert(None)
own_response["period"] = np.where(
    own_response["timestamp"] < MOVE_DATE, "before move", "after move"
)

overnight_excluded = own_response["response_seconds"].gt(MAX_RESPONSE_MINUTES * 60).sum()
responses = own_response[
    own_response["response_seconds"].gt(0)
    & own_response["response_seconds"].le(MAX_RESPONSE_MINUTES * 60)
].copy()
responses["response_minutes"] = responses["response_seconds"] / 60

print(
    f"{overnight_excluded:,} responses over {MAX_RESPONSE_MINUTES / 60:.0f}h excluded "
    "(treated as asleep, not a slow reply)"
)
responses.groupby(["author", "period"])["response_minutes"].count()

In [ ]:
# Colour = person (grey-first isn't the point here -- no group is "the finding" yet,
# just a first look), linestyle = period, so before/after reads the same way for both
# people without needing four colours.
GROUP_STYLE = {
    ("Roos", "before move"): {"color": "#0072B2", "linestyle": "--"},
    ("Roos", "after move"): {"color": "#0072B2", "linestyle": "-"},
    ("Louis", "before move"): {"color": "#D55E00", "linestyle": "--"},
    ("Louis", "after move"): {"color": "#D55E00", "linestyle": "-"},
}

ecdf_settings = PlotSettings(
    figsize=(10, 5),
    title="Response time: Roos vs. Louis, before vs. after the move to NL",
    xlabel="response time (minutes)",
    ylabel="share of responses at or below",
)
host = ECDFPlot(ecdf_settings)
fig = ax = None
for i, ((author, period), style) in enumerate(GROUP_STYLE.items()):
    subset = responses.loc[
        (responses["author"] == author) & (responses["period"] == period), "response_minutes"
    ].to_numpy()
    label = f"{author}, {period}"
    if i == 0:
        fig, ax = host.plot(data=subset, label=label, **style)
    else:
        host.plot_on(ECDFPlot(ecdf_settings), data=subset, label=label, **style)
ax.legend(loc="lower right")

counts = responses.groupby(["author", "period"])["response_minutes"].count()
n_caption = "   ".join(f"{author}, {period}: n={counts[(author, period)]:,}" for author, period in GROUP_STYLE)
fig.text(0.5, -0.08, n_caption, ha="center", fontsize=8.5, color="dimgray")

## 4.6.5 What hour do we talk? Before vs. after the move

Not an event-discovery or pattern-discovery question — the event (`MOVE_DATE`) and the
prediction are both already named: a long-distance "good morning" / "sleep well" ritual
at wake-up and bedtime, before the move, that a shared apartment makes unnecessary after
it. That makes this a direct before/after comparison, same shape as 4.6.4, on a new
variable: which **hour** each message was sent in.

Two choices, both flagged rather than silent:

- **Night hours (00:00-04:59) excluded** — matches the "morning starts at 05:00" boundary
  already in `analysis-log.md`'s time-of-day buckets, so this reuses a definition you
  already settled on rather than inventing a new cutoff.
- **Share of each period's own messages, not raw counts** — the "before" and "after"
  periods have very different total volumes (§4.6.4 already showed that), and the five
  families chapter's own categories warning applies here too: proportions across unequal
  group sizes, or histograms with different sample sizes, mislead unless normalised.

**My prediction, before looking:** more messages in the evening before the move; more
messages during the day after the move.

In [ ]:
NIGHT_HOURS = range(0, 5)  # 00:00-04:59, matches analysis-log.md's morning-starts-at-05:00

with_hour = Pipeline().add(TimeFeatures, column="timestamp", features=["hour"]).apply(own_response)
daytime = with_hour[~with_hour["hour"].isin(NIGHT_HOURS)].copy()

hour_counts = daytime.groupby(["period", "hour"]).size().rename("n").reset_index()
hour_counts["share"] = hour_counts.groupby("period")["n"].transform(lambda s: s / s.sum())

hour_settings = PlotSettings(
    figsize=(13, 5),
    title='What hour do we talk? (night hours 00:00-04:59 excluded)',
    xlabel="hour of day",
    ylabel="share of that period's messages",
)
fig, ax = BarPlot(hour_settings).plot(
    data=hour_counts, x="hour", y="share", hue="period",
    hue_order=["before move", "after move"],
    palette={"before move": hour_settings.base_color, "after move": hour_settings.highlight_color},
)
ax.legend(title="")

n_caption = "   ".join(
    f"{period}: n={int(daytime[daytime.period == period].shape[0]):,}"
    for period in ["before move", "after move"]
)
fig.text(0.5, -0.1, n_caption, ha="center", fontsize=8.5, color="dimgray")

Read against the prediction above: evening vs. daytime hours, before vs. after — did the
weight shift the way you expected, and by how much?

**What I found:** (write it here)

## 4.6.6 Message length and burst length, before vs. after Benji

**Hypothesis, in your words**: since Benji arrived (11 May 2025) you send longer messages
and more bursts than Louis, and the gap between you two is bigger after the baby than
before.

This is the first place in the notebook that actually fits a family rather than just
looking at shape — `goad_distribution_method` steps 5-9, not just 1-4. That matters for
more than method: the graded rubric for this theme is explicit that "welke familie past,
en waarom hoort dat mechanisme bij jouw chat" is the actual ask — a histogram or ECDF
without a fitted family and a stated mechanism is "een plaatje, geen verhaal."

Two variables, one new date:

- **`BABY_DATE = "2025-05-11"`** — a different split than `MOVE_DATE` above; this section
  doesn't touch the move-based `period` column from 4.6.4/4.6.5.
- **Message length** — `n_chars`, already on `own`.
- **Burst length** — your choice: consecutive messages from the same author before the
  other one replies, i.e. a run-length on `author`. One number per *burst*, not per
  message — a 5-message burst is one observation, not five.

In [ ]:
own["baby_period"] = np.where(
    own["timestamp"].dt.tz_convert(None) < BABY_DATE, "before baby", "after baby"
)

length_fitter = DistributionFitter(DistributionRegistry(), seed=42)
length_fits = {}
for author in ["Roos", "Louis"]:
    for period in ["before baby", "after baby"]:
        values = own.loc[
            (own["author"] == author) & (own["baby_period"] == period), "n_chars"
        ].to_numpy().astype(float)
        fits = length_fitter.fit(values, discrete=False)
        best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
        length_fits[(author, period)] = (values, best)
        print(
            f"{author:6s} {period:11s}  {best.distribution:8s}  "
            f"mean {best.frozen_dist.mean():6.1f}  sd {best.frozen_dist.std():6.1f}  "
            f"(n={len(values):,})"
        )

In [ ]:
BABY_GROUP_STYLE = {
    ("Roos", "before baby"): {"color": "#0072B2", "linestyle": "--"},
    ("Roos", "after baby"): {"color": "#0072B2", "linestyle": "-"},
    ("Louis", "before baby"): {"color": "#D55E00", "linestyle": "--"},
    ("Louis", "after baby"): {"color": "#D55E00", "linestyle": "-"},
}

length_settings = PlotSettings(
    figsize=(10, 5),
    title="Message length: Roos vs. Louis, before vs. after Benji",
    xlabel="characters per message",
    ylabel="share of messages at or below",
)
host = ECDFPlot(length_settings)
fig = ax = None
for i, ((author, period), style) in enumerate(BABY_GROUP_STYLE.items()):
    values, best = length_fits[(author, period)]
    label = f"{author}, {period} ({best.distribution}, mean {best.frozen_dist.mean():.0f})"
    if i == 0:
        fig, ax = host.plot(data=values, label=label, **style)
    else:
        host.plot_on(ECDFPlot(length_settings), data=values, label=label, **style)
ax.legend(loc="lower right", fontsize=8.5)

n_caption = "   ".join(
    f"{author}, {period}: n={len(length_fits[(author, period)][0]):,}"
    for author, period in BABY_GROUP_STYLE
)
fig.text(0.5, -0.1, n_caption, ha="center", fontsize=8.5, color="dimgray")

**Why that family, if `lognorm` wins** (fill in against what the printout above actually
says): a message's length isn't a sum of independent contributions, it's closer to a
*product* — how much you have to say, multiplied by how rushed you are, multiplied by
whether you're mid-conversation or opening one, multiplied by topic complexity. 04.2 made
the identical argument for taxi fares (base fare × distance × traffic × surcharges) and it
predicted `lognorm` there too — the mechanism, not just the shape, is what makes this
prediction more than a coincidence. If a different family won instead, that mechanism
story needs to change to match it, not the other way around.

**Burst length next** — same treatment, but discrete: a *count* of consecutive messages,
not a continuous measurement.

In [ ]:
# One row per burst, not per message: consecutive same-author messages collapse into
# one run, counted once.
own_sorted = own.sort_values("timestamp").reset_index(drop=True)
run_id = (own_sorted["author"] != own_sorted["author"].shift()).cumsum()
bursts = (
    own_sorted.groupby(run_id)
    .agg(
        author=("author", "first"),
        timestamp=("timestamp", "first"),
        burst_length=("author", "size"),
    )
    .reset_index(drop=True)
)
bursts["baby_period"] = np.where(
    bursts["timestamp"].dt.tz_convert(None) < BABY_DATE, "before baby", "after baby"
)

burst_fitter = DistributionFitter(DistributionRegistry(), seed=42)
burst_fits = {}
for author in ["Roos", "Louis"]:
    for period in ["before baby", "after baby"]:
        values = bursts.loc[
            (bursts["author"] == author) & (bursts["baby_period"] == period), "burst_length"
        ].to_numpy().astype(float)
        fits = burst_fitter.fit(values, discrete=True)
        best = next(f for f in fits if isinstance(f, FitResult) and f.best_likelihood)
        burst_fits[(author, period)] = (values, best)
        print(
            f"{author:6s} {period:11s}  {best.distribution:8s}  "
            f"mean {best.frozen_dist.mean():5.2f}  sd {best.frozen_dist.std():5.2f}  "
            f"(n={len(values):,} bursts)"
        )

**How a burst is actually computed** — walking through the cell above:

1. **Sort by timestamp.** Order matters here: a burst is about who talked *in a row*.
2. **Mark every author-change.** `own_sorted["author"] != own_sorted["author"].shift()`
   is `True` exactly on the message where the author switches from the one before it.
3. **`.cumsum()` turns each change into the start of a new run.** A toy example makes
   this concrete — same mechanic, eight messages instead of 63,000:

   | # | author | changed from previous? | `run_id` (cumsum) |
   |---|---|---|---|
   | 1 | Roos | — (first row) | 1 |
   | 2 | Roos | False | 1 |
   | 3 | Louis | **True** | 2 |
   | 4 | Roos | **True** | 3 |
   | 5 | Roos | False | 3 |
   | 6 | Roos | False | 3 |
   | 7 | Louis | **True** | 4 |
   | 8 | Louis | False | 4 |

   Four runs: Roos×2, Louis×1, Roos×3, Louis×2. `run_id` only increases where the
   `True`/`False` column is `True` — everywhere else it repeats the same number, which
   is exactly what "still the same burst" means.
4. **`.groupby(run_id).agg(...)` collapses each run into one row**: which author, when
   it started (first timestamp — used for the before/after split), and `burst_length` =
   how many messages share that `run_id` (`.size()`).

One burst = one observation, whatever its length — a 7-message monologue counts once,
the same as a single lone message (`burst_length = 1`). That's why `bursts` has far
fewer rows (~12,500) than `own` (~63,000 messages): most bursts are more than one
message.

In [ ]:
burst_settings = PlotSettings(
    figsize=(10, 5),
    title="Burst length: Roos vs. Louis, before vs. after Benji",
    xlabel="messages per burst",
    ylabel="share of bursts at or below",
)
host = ECDFPlot(burst_settings)
fig = ax = None
for i, ((author, period), style) in enumerate(BABY_GROUP_STYLE.items()):
    values, best = burst_fits[(author, period)]
    label = f"{author}, {period} ({best.distribution}, mean {best.frozen_dist.mean():.2f})"
    if i == 0:
        fig, ax = host.plot(data=values, label=label, **style)
    else:
        host.plot_on(ECDFPlot(burst_settings), data=values, label=label, **style)
ax.legend(loc="lower right", fontsize=8.5)

n_caption = "   ".join(
    f"{author}, {period}: n={len(burst_fits[(author, period)][0]):,} bursts"
    for author, period in BABY_GROUP_STYLE
)
fig.text(0.5, -0.1, n_caption, ha="center", fontsize=8.5, color="dimgray")

**Why that family, if `nbinom` wins** (again: fill this in against the actual printout,
not this guess): a burst is "how many messages before the other person gets a word in" —
each message in the run either continues (you keep typing) or stops (you wait for a
reply), which is the same steady-continuation-probability mechanism 04.4 §4.4.1 used for
`nbinom` over `poisson`: not one fixed rate of continuing, but a rate that itself varies
by conversation, mood, or topic — a "sometimes I fire off ten in a row, sometimes just
one" pattern rather than a metronomic one. If `poisson` or something else wins instead,
that says the continuation rate is steadier than this story assumes, and the story should
say so.

**What I found:** (read both charts against your hypothesis — longer messages *and* more
bursts for you than Louis, with a bigger gap after Benji — write it here, including
anywhere the fitted family surprised you)

**Still open, if you want this to hold up under `spurious-correlations`**: everything
above is a fit and a picture, not yet a "how sure" check — `NullDistribution` (04.4
§4.4.3) would say whether the before/after gap in either variable is bigger than shuffling
the labels would produce on its own. Worth doing before this goes in the report.

## 4.6.7 How sure? A `NullDistribution` check on all four charts

Same idea for all four, so it's written once here rather than per chart: every comparison
above is "before vs. after, Roos vs. Louis" — a gap between two people that might itself
change between two periods. The statistic each check below measures is that **gap of
gaps** (also called an interaction, or difference-in-differences):

```
(Roos, after) − (Louis, after)   −   (Roos, before) − (Louis, before)
```

`NullDistribution` then shuffles the *before/after* label 2,000 times — keeping who-said-
what and each period's proportions exactly as they were, only breaking the link between
"when" and the value — and recomputes that same gap-of-gaps each time. If the real
gap-of-gaps sits far outside that shuffled cloud, "before/after" is doing real work; if it
sits comfortably inside, a random split would have produced a gap this big just as often,
and "there might be a switch" is not yet a finding.

`hour-of-day` has no author split (4.6.5 never had one), so its check is simpler: evening
share (hours 18:00-23:59, approximating the 17:30 boundary to whole hours) after the move
minus before it — no gap-of-gaps, just whether that shift is more than shuffle-noise.

In [ ]:
def response_gap_change(frame: pd.DataFrame) -> float:
    means = frame.groupby(["author", "period"])["response_minutes"].mean()
    return (means[("Roos", "after move")] - means[("Louis", "after move")]) - (
        means[("Roos", "before move")] - means[("Louis", "before move")]
    )


def evening_share_change(frame: pd.DataFrame) -> float:
    is_evening = frame["hour"].isin(EVENING_HOURS)
    shares = is_evening.groupby(frame["period"]).mean()
    return shares["after move"] - shares["before move"]


def length_gap_change(frame: pd.DataFrame) -> float:
    means = frame.groupby(["author", "baby_period"])["n_chars"].mean()
    return (means[("Roos", "after baby")] - means[("Louis", "after baby")]) - (
        means[("Roos", "before baby")] - means[("Louis", "before baby")]
    )


def burst_gap_change(frame: pd.DataFrame) -> float:
    means = frame.groupby(["author", "baby_period"])["burst_length"].mean()
    return (means[("Roos", "after baby")] - means[("Louis", "after baby")]) - (
        means[("Roos", "before baby")] - means[("Louis", "before baby")]
    )


EVENING_HOURS = range(18, 24)

response_null = NullDistribution(response_gap_change, n_iter=2000, seed=4).run(responses, label="period")
hour_null = NullDistribution(evening_share_change, n_iter=2000, seed=4).run(daytime, label="period")
length_null = NullDistribution(length_gap_change, n_iter=2000, seed=4).run(own, label="baby_period")
burst_null = NullDistribution(burst_gap_change, n_iter=2000, seed=4).run(bursts, label="baby_period")

for name, result in [
    ("response time gap-of-gaps (minutes)", response_null),
    ("evening-share shift", hour_null),
    ("message-length gap-of-gaps (chars)", length_null),
    ("burst-length gap-of-gaps (messages)", burst_null),
]:
    print(f"{name:38s} observed {result.observed:+7.3f}   two-sided p = {result.p_value():.4f}")

In [ ]:
null_settings = PlotSettings(
    figsize=(14, 9),
    title="Is the before/after gap bigger than shuffled labels would produce?",
    subplot_titles=[
        "Response time: (Roos−Louis) gap, after − before move",
        "Evening share (18:00-23:59), after − before move",
        "Message length: (Roos−Louis) gap, after − before baby",
        "Burst length: (Roos−Louis) gap, after − before baby",
    ],
    # Each panel's x-axis is the STATISTIC that got shuffled 2,000 times, not the raw
    # variable (not "minutes" or "characters" per message) -- naming that explicitly
    # here is the fix for "what is this axis even showing".
    subplot_xlabels=[
        "gap-of-gaps (minutes), one value per shuffle",
        "evening-share shift, one value per shuffle",
        "gap-of-gaps (characters), one value per shuffle",
        "gap-of-gaps (messages/burst), one value per shuffle",
    ],
    ylabel="number of shuffles landing here",
    max_cols=2,
)
host = NullPlot(null_settings)
fig, axes = host.create_figure(n_plots=4)
for ax, result in zip(axes, [response_null, hour_null, length_null, burst_null]):
    host.plot_on_axes(NullPlot(null_settings), ax, result=result)
    ax.legend(fontsize=8)

Read each panel the same way: grey cloud = what shuffling before/after produces on its
own; crimson line = what actually happened. Far outside the cloud = solid; inside it =
could be chance. The printed p-values above give the same read as a number — under 0.05
is a common (not sacred) line, and with 2,000 shuffles anything reported as "p < 0.0005"
is really "rarer than this test can resolve," not exactly zero.

**What I found:** (write it here, one line per variable — which of the four survived this
check, and which turned out to be a gap you can't distinguish from noise)

## 4.6.8 Burst length, Roos vs. Louis, side by side

Two panels instead of one ECDF: one per person, each with its own before/after fitted
curves overlaid. "Louis has fewer bursts than Roos" now reads as two panels sitting at
different heights; "Roos's bursts got longer after Benji" reads within her own panel —
both without a four-line legend to trace on one shared axis. `sharex`/`sharey` keep both
panels on the same scale, so comparing their heights directly is honest.

Curves, not histogram bars — the fit is already verified (4.6.6), and two overlapping
discrete histograms are exactly the clutter §7.3 of the visual-critique chapter warns
about; the smooth `pmf` curves are the same information with the mess removed.

In [ ]:
burst_compare_settings = PlotSettings(
    figsize=(12, 5),
    title="Mom takes the lead",
    subplot_titles=["Roos", "Louis"],
    xlabel="messages per burst",
    ylabel="probability",
    max_cols=2,
    sharex=True,
    sharey=True,
)
host = DistPlot(burst_compare_settings)
fig, axes = host.create_figure(n_plots=2)
for ax, author in zip(axes, ["Roos", "Louis"]):
    _, before_fit = burst_fits[(author, "before baby")]
    _, after_fit = burst_fits[(author, "after baby")]
    host.plot_on_axes(
        DistPlot(burst_compare_settings), ax, distribution=before_fit.frozen_dist,
        color=burst_compare_settings.base_color, marker="o",
        label=f"before baby ({before_fit.distribution}, mean {before_fit.frozen_dist.mean():.1f})",
    )
    host.plot_on_axes(
        DistPlot(burst_compare_settings), ax, distribution=after_fit.frozen_dist,
        color=burst_compare_settings.highlight_color, marker="o",
        label=f"after baby ({after_fit.distribution}, mean {after_fit.frozen_dist.mean():.1f})",
    )
    ax.legend(fontsize=8)

## 4.6.9 The real finding: not *more*, but *opposite directions*

Read against 4.6.8: the overall shift for either person is small, and that's the point
you're actually making — not "bigger bursts," but "yours grew and his shrank." Neither
the ECDF (4.6.6) nor the side-by-side panels (4.6.8) put that on the page directly; both
still ask the reader to compare two shapes and notice the direction themselves.

This chart does the subtraction for them — the same move 03.1 used on a time series
(`SubtractBaseline`: after minus before, so the residual is the plot), applied here to
two *fitted distributions* instead of two time windows. For each possible burst length,
`after.pmf(x) − before.pmf(x)`: positive means that length became *more* likely after
Benji, negative means *less* likely. One line per person, one shared zero line — Roos's
line should sit above zero further right (bigger bursts, more likely), Louis's below.

New class in `scripts/plots.py`: `DistributionDiffPlot` — draws a precomputed
`(x, diff)` pair as a line with the area shaded to the zero line, so the sign reads as
much from the fill as from the line itself.

In [ ]:
DIFF_COLORS = {"Roos": "#0072B2", "Louis": "#D55E00"}

# Shared x-range: covers the 99th percentile of every fitted distribution, so neither
# person's curve gets cut off, and both are drawn on the same integers for a fair
# subtraction.
max_x = max(
    int(np.ceil(fit.frozen_dist.ppf(0.99)))
    for _, fit in burst_fits.values()
)
x = np.arange(1, max_x + 1)

diff_settings = PlotSettings(
    figsize=(10, 5),
    title="Mom takes the lead",
    xlabel="messages per burst",
    ylabel="P(after Benji) − P(before Benji)",
)
host = DistributionDiffPlot(diff_settings)
fig = ax = None
for i, author in enumerate(["Roos", "Louis"]):
    _, before_fit = burst_fits[(author, "before baby")]
    _, after_fit = burst_fits[(author, "after baby")]
    diff = after_fit.frozen_dist.pmf(x) - before_fit.frozen_dist.pmf(x)
    if i == 0:
        fig, ax = host.plot(x=x, diff=diff, color=DIFF_COLORS[author], label=author)
    else:
        host.plot_on(DistributionDiffPlot(diff_settings), x=x, diff=diff,
                     color=DIFF_COLORS[author], label=author)
host.plot_on(HorizontalLine(diff_settings), y=0)
ax.legend()

fig.text(
    0.5, -0.08,
    "Positive = that burst length became more likely after Benji; negative = less "
    "likely. Fitted pmf difference, not raw counts.",
    ha="center", fontsize=8.5, color="dimgray",
)